#### Get Marker Genes for All Datasets

In [23]:
import anndata as ad
adata = ad.read_h5ad('/work/magroup/skrieger/Syndri/data/datasets/breast_xenium/interim/single-cell/breast_cancer_chromium_genesubset.h5ad')
adata

AnnData object with n_obs × n_vars = 10658 × 277
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'sample_id', 'percent_mt', 'percent_ribo', 'percent_hb', 'nCount_SCT', 'nFeature_SCT', 'SCT_snn_res.0.8', 'seurat_clusters', 'SCT_snn_res.0.5', 'SCT_snn_res.0.4', 'SCT_snn_res.2', 'donor_id', 'nCount_SoupX', 'nFeature_SoupX', 'Harmonised_Level4', 'Level3', 'Level2', 'Level1', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'disease_ontology_term_id', 'tissue_type', 'tissue_ontology_term_id', 'cell_type_ontology_term_id', 'assay_ontology_term_id', 'suspension_type', 'input_material', 'histology', 'sample_type', 'histology_immune', 'DV200_percent', 'block_age_months', 'is_primary_data', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns:

In [ ]:
import pandas as pd
gene_annot = pd.read_excel('/work/magroup/skrieger/Syndri/data/datasets/breast_xenium/raw/single-cell/41592_2026_3089_MOESM4_ESM.xlsx',sheet_name='ST 4',skiprows=1)
gene_annot
adata.var = adata.var.merge(gene_annot,left_index=True,right_on='Ensembl_ID')
adata.var = adata.var.set_index('Ensembl_ID')

In [ ]:
sorted(adata_ext.obs['celltype_subset'].unique())

['B cells Memory',
 'B cells Naive',
 'CAFs MSC iCAF-like s1',
 'CAFs MSC iCAF-like s2',
 'CAFs Transitioning s3',
 'CAFs myCAF like s4',
 'CAFs myCAF like s5',
 'Cancer Basal SC',
 'Cancer Cycling',
 'Cancer Her2 SC',
 'Cancer LumA SC',
 'Cancer LumB SC',
 'Cycling PVL',
 'Cycling_Myeloid',
 'Endothelial ACKR1',
 'Endothelial CXCL12',
 'Endothelial Lymphatic LYVE1',
 'Endothelial RGS5',
 'Luminal Progenitors',
 'Mature Luminal',
 'Myeloid_c0_DC_LAMP3',
 'Myeloid_c10_Macrophage_1_EGR1',
 'Myeloid_c11_cDC2_CD1C',
 'Myeloid_c12_Monocyte_1_IL1B',
 'Myeloid_c1_LAM1_FABP5',
 'Myeloid_c2_LAM2_APOE',
 'Myeloid_c3_cDC1_CLEC9A',
 'Myeloid_c4_DCs_pDC_IRF7',
 'Myeloid_c5_Macrophage_3_SIGLEC1',
 'Myeloid_c7_Monocyte_3_FCGR3A',
 'Myeloid_c8_Monocyte_2_S100A9',
 'Myeloid_c9_Macrophage_2_CXCL10',
 'Myoepithelial',
 'PVL Differentiated s3',
 'PVL Immature s1',
 'PVL_Immature s2',
 'Plasmablasts',
 'T_cells_c0_CD4+_CCR7',
 'T_cells_c10_NKT_cells_FCGR3A',
 'T_cells_c11_MKI67',
 'T_cells_c1_CD4+_IL7R',
 

In [ ]:
sorted(adata.obs['celltype_minor'].unique())

['CD4-positive, alpha-beta memory T cell',
 'capillary endothelial cell',
 'classical monocyte',
 'cytotoxic T cell',
 'fibroblast of breast',
 'fibroblast of lung',
 'macrophage',
 'malignant cell',
 'mast cell',
 'natural T-regulatory cell',
 'natural killer cell',
 'pericyte',
 'plasma cell',
 'plasmacytoid dendritic cell',
 'smooth muscle cell']

In [ ]:
adata.obs.cell_type

B1_AAACAAGCAGCCTTCAACTTTAGG-1         natural T-regulatory cell
B1_AAACAAGCAGGAATAAACTTTAGG-1                       plasma cell
B1_AAACAAGCATTTGGAGACTTTAGG-1                  cytotoxic T cell
B1_AAACCAATCACGTTCAACTTTAGG-1                    malignant cell
B1_AAACCAATCGTCAATTACTTTAGG-1                       plasma cell
                                                ...            
B4_rep_TTTGCGCAGTGAATCAATGTTGAC-1            smooth muscle cell
B4_rep_TTTGCTCTCCATCCTTATGTTGAC-1                      pericyte
B4_rep_TTTGCTGAGATAGCCGATGTTGAC-1            smooth muscle cell
B4_rep_TTTGGACGTTGGTTACATGTTGAC-1                malignant cell
B4_rep_TTTGGCGGTGCGCATCATGTTGAC-1    capillary endothelial cell
Name: cell_type, Length: 10658, dtype: category
Categories (15, object): ['mast cell', 'smooth muscle cell', 'macrophage', 'natural killer cell', ..., 'malignant cell', 'capillary endothelial cell', 'fibroblast of lung', 'fibroblast of breast']

In [ ]:
import numpy as np
import pandas as pd
from scipy import sparse

def pseudobulk_one_vs_rest(
    adata,
    target,
    sample_col="sample",
    cell_type_col="cell_type",
    layer="counts",
    min_cells=20,
):
    X = adata.layers[layer] if layer is not None else adata.X

    rows = []
    metadata = []

    for sample in adata.obs[sample_col].unique():

        sample_mask = adata.obs[sample_col].values == sample
        target_mask = (
            sample_mask
            & (adata.obs[cell_type_col].values == target)
        )
        rest_mask = (
            sample_mask
            & (adata.obs[cell_type_col].values != target)
        )

        # Avoid pseudobulks based on only a handful of cells
        if target_mask.sum() < min_cells or rest_mask.sum() < min_cells:
            continue

        for group, mask in [
            ("target", target_mask),
            ("rest", rest_mask),
        ]:
            counts = X[mask].sum(axis=0)

            if sparse.issparse(counts):
                counts = counts.A1
            else:
                counts = np.asarray(counts).ravel()

            rows.append(counts)

            metadata.append({
                "sample": str(sample),
                "group": group,
            })

    sample_names = [
        f"{m['sample']}_{m['group']}"
        for m in metadata
    ]

    counts_df = pd.DataFrame(
        np.vstack(rows),
        index=sample_names,
        columns=adata.var_names,
    ).astype(int)

    metadata_df = pd.DataFrame(
        metadata,
        index=sample_names,
    )

    return counts_df, metadata_df